# Scenario Playground Step 10: Naive and Ridge Baselines

Objective: train one reproducible Ridge model for each of the 12 approved targets, generate the first out-of-sample predictions on the validation period, and compare Ridge with zero, development-training mean, and target-consistent recent-trailing baselines.

Safety boundary: preprocessing and Ridge are fit on `development_train` only. The matrix reader stops before the locked final-test rows, so their target values are neither parsed nor used for fitting, alpha selection, or reporting.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.ridge_baselines import (
    RIDGE_ALPHAS,
    build_ridge_artifacts,
    persist_ridge_artifacts,
)

artifacts = build_ridge_artifacts()
output_paths = persist_ridge_artifacts(artifacts)
summary = artifacts["summary"]
summary["training_gate"], output_paths

## Frozen data and lock guard

The approved matrix contains 584 rows and 58 features. Step 10 parses only its first 470 chronological rows: 354 retained development rows, 110 retained validation rows, and six boundary-purge rows that remain excluded. The remaining 114 final-test rows are not parsed.

In [ ]:
guard = pd.Series(summary["matrix_guard"], name="value")
display(guard.to_frame())
assert summary["feature_count"] == 58
assert summary["target_count"] == 12
assert summary["development_fit_rows"] == 354
assert summary["validation_rows"] == 110
assert summary["matrix_guard"]["boundary_purge_rows_parsed_but_not_used"] == 6
assert summary["matrix_guard"]["locked_final_test_target_rows_parsed"] == 0
assert not summary["matrix_guard"]["final_test_target_values_available_to_step_10"]

## Train-only pipeline and alpha selection

Every target uses `SimpleImputer(strategy='median') -> StandardScaler() -> Ridge()`. All transformations are inside the pipeline and fit from the 354 development rows. Each target selects its alpha from the declared grid by validation MAE, then RMSE and smaller alpha as deterministic tie-breakers. These validation-selected results are not a final unbiased test estimate.

In [ ]:
selected = artifacts["selected_configurations"]
selected_view = selected[[
    "target", "unit", "selected_alpha", "validation_mae", "validation_rmse",
    "validation_oos_r_squared", "validation_prediction_actual_correlation",
    "validation_directional_accuracy",
]]
display(selected_view.round(4))
assert len(selected) == 12
assert len(artifacts["alpha_search"]) == 12 * len(RIDGE_ALPHAS)
assert selected["development_fit_rows"].eq(354).all()
assert selected["validation_selection_rows"].eq(110).all()

## Ridge versus three naive baselines

The recent-trailing baseline uses the same source and output formula as each target over the prior 21 nonmissing source observations ending on that row's target-baseline date. It does not use 21 event-aware sample rows.

In [ ]:
comparison = artifacts["comparison"]
display(comparison[[
    "target", "ridge_mae", "best_naive_mae", "best_naive_by_mae",
    "ridge_mae_improvement_pct", "ridge_beats_best_naive_mae",
    "ridge_rmse", "best_naive_rmse", "best_naive_by_rmse",
    "ridge_rmse_improvement_pct", "ridge_beats_best_naive_rmse",
]].round(4))
print("Ridge beats best naive MAE:", int(comparison["ridge_beats_best_naive_mae"].sum()), "/ 12")
print("Ridge beats best naive RMSE:", int(comparison["ridge_beats_best_naive_rmse"].sum()), "/ 12")

## Full validation metrics and period stability

MAE and RMSE retain each target's native output unit: percentage points for 11 return targets and basis points for the 10-year-yield target. OOS R-squared compares against the validation-mean denominator; correlation is undefined for truly constant baseline predictions.

In [ ]:
metrics = artifacts["metrics"]
metric_pivot = metrics.pivot(index="target", columns="strategy", values=["mae", "rmse"])
display(metric_pivot.round(4))
ridge_years = artifacts["metrics_by_year"].query("strategy == 'ridge'")
display(ridge_years[["target", "calendar_year", "sample_count", "mae", "rmse", "directional_accuracy"]].round(4))
assert len(metrics) == 48
assert len(ridge_years) == 36

## Preprocessing, trailing-history, and coefficient audits

The audit tables preserve fitted medians/scaler statistics, each target-specific trailing observation window, and standardized Ridge coefficients. They support verification and later interpretation without exporting production model objects.

In [ ]:
preprocessing = artifacts["preprocessing_audit"]
trailing = artifacts["trailing_baseline_audit"]
coefficients = artifacts["coefficients"]
display(preprocessing.head())
display(trailing.head())
top_coefficients = (
    coefficients.assign(abs_coefficient=coefficients["standardized_coefficient"].abs())
    .sort_values(["target", "abs_coefficient"], ascending=[True, False])
    .groupby("target", sort=False)
    .head(5)
)
display(top_coefficients[["target", "feature", "standardized_coefficient"]])
assert len(preprocessing) == 12 * 58
assert len(coefficients) == 12 * 58
assert len(trailing) == 12 * 110
assert trailing["point_in_time_pass"].all()

## Acceptance result

Step 10 is complete when every implementation check passes. Weak validation performance is retained honestly: it is model evidence for Step 11, not a reason to alter the approved data or unlock the final test.

In [ ]:
validation_checks = pd.Series(summary["validations"], name="passed")
display(validation_checks.to_frame())
assert validation_checks.all()
assert summary["pipeline"]["fit_role"] == "development_train"
assert not summary["pipeline"]["validation_used_in_fit"]
assert not summary["pipeline"]["locked_final_test_used_in_fit_or_selection"]
assert summary["training_gate"] == "STEP_10_RIDGE_COMPLETE_READY_FOR_STEP_11_XGBOOST_PROTOTYPE"
print(summary["training_gate"])